# Python and Market Data Fundamentals

Audience: Python learners preparing for quant interviews. Prerequisites: basic Python and pandas.

Goals: Lists, dictionaries, DataFrames, time series, and validation.

Outline: setup, short tasks, checks, pitfalls. Run cells from top to bottom. All market data here is synthetic.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from market_simulator import (MarketMaker, ClientModel, MarketEnvironment,
    QuotedTrade, CompletedTrade, generate_market_data, generate_rfqs, run_simulation, make_quote)
from market_simulator.risk import beta, historical_volatility

# Resolve synthetic data from the installed local package, independent of kernel CWD.
import market_simulator
DATA = Path(market_simulator.__file__).resolve().parents[1] / 'data'
prices = pd.read_csv(DATA / 'market_prices.csv', parse_dates=['timestamp'])
requests = pd.read_csv(DATA / 'client_rfqs.csv', parse_dates=['timestamp'])
print(prices.head(2).to_string(index=False))


                timestamp ticker   mid  market_bid  market_ask
2026-01-02 09:30:00+00:00   ALFA 100.0        99.9       100.1
2026-01-02 09:30:00+00:00   BETA 100.0        99.9       100.1


## Sample requests

Return the first n records as dictionaries in original order. For n <= 0, return an empty list. Example: [{'ticker':'A'}, {'ticker':'B'}], 1 -> [{'ticker':'A'}].


In [5]:
def sample_requests(records: list[dict], n: int) -> list[dict]:
    # TODO
    return records[:n]

print(sample_requests(prices, 2))
print(sample_requests([], 1))

                  timestamp ticker    mid  market_bid  market_ask
0 2026-01-02 09:30:00+00:00   ALFA  100.0        99.9       100.1
1 2026-01-02 09:30:00+00:00   BETA  100.0        99.9       100.1
[]


## Validate market data

Return True only if required columns timestamp, ticker, mid exist, every mid is finite and positive, and (timestamp,ticker) keys are unique. Empty rows are allowed if columns exist.


In [7]:
def valid_market(frame: pd.DataFrame) -> bool:
    # TODO
    need = {'timestamp', 'ticker', 'mid'}
    return (need.issubset(frame.columns) and frame['mid'].notna().all() 
        and (frame['mid'] > 0).all() and np.isfinite(frame['mid']).all()
        and frame['ticker'].notna().all() and frame['timestamp'].notna().all()
        and not frame.duplicated(['timestamp', 'ticker']).any())

print(valid_market(prices))
print(valid_market(pd.DataFrame({'timestamp':[1,1], 'ticker':['A','A'], 'mid':[1,2]})))

True
False


## Group and return

For one ticker, sort by timestamp and return a Series of simple returns (P_t/P_(t-1)-1), excluding the first missing value. Example prices 100, 110 -> [0.10].


In [14]:
def ticker_returns(frame: pd.DataFrame, ticker: str) -> pd.Series:
    # TODO
    series = frame.loc[frame['ticker'] == ticker].sort_values('timestamp')['mid']
    return series.pct_change().dropna()

print(ticker_returns(pd.DataFrame({'ticker':['A','A'], 'timestamp':[1,2], 'mid':[100,110]}), 'A'))
print(ticker_returns(prices, 'MISSING').empty)

1    0.1
Name: mid, dtype: float64
True


## Extension
Try a different random seed or add a transaction fee. Explain how your answer changes.